# Graph Memory — Reasoning Over Relationships, Not Just Similarity

Semantic memory (Demo 03) retrieves by *similarity* but can't reason over *relationships*.
A multi-hop question needs to find an entry point by similarity, then **traverse the graph**.

**The question this demo answers:** *"Who did I meet that's connected to boutique hotels in Japan?"*

Based on research:
- [GAAMA: Graph Augmented Associative Memory for Agents](https://arxiv.org/abs/2603.27910) — Paul et al., 2026
- [MAGMA: A Multi-Graph based Agentic Memory Architecture for AI Agents](https://arxiv.org/abs/2601.03236) — Jiang et al., 2026
- [GRAVITY: Architecture-Agnostic Structured Anchoring for Long-Horizon Conversational Memory](https://arxiv.org/abs/2605.01688) — Sun et al., 2026

Uses [Strands Agents](https://github.com/strands-agents/sdk-python) for the harness and
[Neo4j](https://neo4j.com/) + [`neo4j-graphrag`](https://neo4j.com/docs/neo4j-graphrag-python/) for graph memory.

## Prerequisites

1. A running **Neo4j** (Desktop, Docker, or Aura).
2. Environment variables in a `.env` file (copy from `.env.example`):
   `OPENAI_API_KEY`, `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`, `NEO4J_DATABASE`.

The demo uses OpenAI by default; swap the model/embeddings for Amazon Bedrock in production (Demo 07).

In [1]:
import os
os.environ['OTEL_SDK_DISABLED'] = 'true'

from dotenv import load_dotenv
load_dotenv()

assert os.getenv('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in your .env'
assert os.getenv('NEO4J_PASSWORD'), 'Set NEO4J_* values in your .env'
print('Config loaded.')

Config loaded.


## Setup — build the graph memory

`graph_memory.build()` connects to Neo4j, ensures an isolated database (and Cypher 25 where needed),
seeds the known graph with real embeddings, and creates the native vector index.

In [2]:
import graph_memory as gm
import travel_tools as tt

# Using OpenAI-compatible interface via Strands SDK (not direct OpenAI usage)
from strands import Agent
from strands.models.openai import OpenAIModel

MODEL = OpenAIModel(model_id='gpt-4o-mini')

driver, db, embedder = gm.build()
QUESTION = gm.MULTIHOP_QUESTION
print('Question:', QUESTION)

  ✅ Database 'memorydemo' uses Cypher 25 (required by the vector retrievers on this server).


  Seeded graph in database 'memorydemo': 5 nodes, 4 relationships, vector index 'memory_embeddings'.
Question: Who did I meet that's connected to boutique hotels in Japan?


## The seeded graph

What the agent learned across sessions, stored as connected nodes:

```
(Sarah Chen) ─WORKS_AT→ (Vista Hotels) ─HAS_STYLE→ (boutique)
                              │
                         LOCATED_IN
                              ▼
                          (Kyoto) ─IN_COUNTRY→ (Japan)
```

The answer to the question — **Sarah Chen** — is never stated; you can only reach it by following edges.

---
## Test 1 — Semantic recall (before)

`VectorRetriever` = pure vector similarity. It surfaces related pieces but can't connect them to a person.

In [3]:
before = gm.make_before_retriever(driver, db, embedder)
result = before.search(query_text=QUESTION, top_k=3)
for item in result.items:
    print(' -', item.content)
print('\nRecovers Sarah Chen?', any('Sarah Chen' in i.content for i in result.items))

 - {'name': 'boutique', 'type': 'Style', 'text': 'boutique. A hotel category.'}
 - {'name': 'Kyoto', 'type': 'Location', 'text': 'Kyoto. A city.'}
 - {'name': 'Japan', 'type': 'Country', 'text': 'Japan. A country.'}

Recovers Sarah Chen? False


---
## Test 2 — Graph recall (after)

`VectorCypherRetriever` = similarity to find an entry node, then a Cypher traversal back to the person.
It returns the full chain.

In [4]:
after = gm.make_after_retriever(driver, db, embedder)
result = after.search(query_text=QUESTION, top_k=3)
for item in result.items:
    print(' -', item.content)
print('\nRecovers Sarah Chen?', any('Sarah Chen' in i.content for i in result.items))

 - <Record who='Sarah Chen' chain=['Sarah Chen', 'Vista Hotels', 'boutique'] score=0.74266117811203>
 - <Record who='Sarah Chen' chain=['Sarah Chen', 'Vista Hotels', 'Kyoto'] score=0.695400595664978>
 - <Record who='Sarah Chen' chain=['Sarah Chen', 'Vista Hotels', 'Kyoto', 'Japan'] score=0.6915712952613831>

Recovers Sarah Chen? True


---
## Test 3 — A full Strands agent with graph memory

The agent uses `recall_graph` to answer, and `remember_fact` to write a new fact back into the graph —
the harness is just tools + state.

In [5]:
tt.init_memory(driver=driver, db=db, embedder=embedder)

agent = Agent(
    model=MODEL,
    system_prompt=(
        'You are a travel assistant with graph memory. Use recall_graph to answer '
        'questions about people and places, and remember_fact to store new durable facts. Be concise.'
    ),
    tools=[tt.recall_graph, tt.recall_semantic, tt.remember_fact],
    callback_handler=None,
)

resp = agent(QUESTION)
print('Agent:', resp.message['content'][0]['text'].strip())

Agent: You met Sarah Chen, who is connected to boutique hotels through Vista Hotels in Kyoto, Japan.


In [6]:
resp = agent('By the way, remember that Sarah Chen works at Vista Hotels — she is my contact there.')
print('Agent:', resp.message['content'][0]['text'].strip())
print('\nFacts logged to agent.state:', agent.state.get('remembered_facts'))

Agent: I've noted that Sarah Chen works at Vista Hotels as your contact there.

Facts logged to agent.state: [{'subject': 'Sarah Chen', 'relation': 'WORKS_AT', 'object': 'Vista Hotels'}]


---
## Test 4 — Deterministic scorecard

Four multi-hop questions, checked against the known graph (no LLM judge). Reproducible.

In [7]:
SCORECARD = [
    ("Who did I meet that's connected to boutique hotels in Japan?", 'Sarah Chen'),
    ('Who did I meet connected to a hotel in Kyoto?', 'Sarah Chen'),
    ('Who works at the boutique hotel brand I know?', 'Sarah Chen'),
    ('Which person is linked to hotels in Japan?', 'Sarah Chen'),
]

before_hits = after_hits = 0
print(f"{'Question':<52}{'before':>8}{'after':>8}")
for q, target in SCORECARD:
    b = any(target in it.content for it in before.search(query_text=q, top_k=3).items)
    a = any(target in it.content for it in after.search(query_text=q, top_k=3).items)
    before_hits += b; after_hits += a
    print(f"{q[:52]:<52}{('OK' if b else '-'):>8}{('OK' if a else '-'):>8}")

print(f'\nCorrect — before: {before_hits}/{len(SCORECARD)} | after: {after_hits}/{len(SCORECARD)}')

Question                                              before   after


Who did I meet that's connected to boutique hotels i       -      OK


Who did I meet connected to a hotel in Kyoto?              -      OK


Who works at the boutique hotel brand I know?             OK      OK


Which person is linked to hotels in Japan?                 -      OK

Correct — before: 1/4 | after: 4/4


---
## Summary

| Retriever | Strategy | Multi-hop answer? |
|-----------|----------|-------------------|
| `VectorRetriever` (before) | Pure similarity | No — finds pieces, can't connect them |
| `VectorCypherRetriever` (after) | Similarity + traversal | Yes — returns the full chain |

**Key insight:** similarity finds related pieces; only traversal connects them. Graph memory answers
multi-hop questions that flat/semantic memory cannot — and with Strands, plugging in the graph store
is just tools + state.

Both retrievers got the **same facts** and shared the **same vector index**. The graph wins structurally,
not because it was handed the answer.

In [8]:
driver.close()
print('Done.')

Done.
